In [ ]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=1

import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import vis_utils

from utils.utils import get_path
from utils.io_utils import load_multiple_res
from utils.toydata_utils import get_toy_data
from utils.fig_utils import dataset_to_print, plot_dgm_loops, dist_to_print
from utils.pd_utils import sort_cycle

from vis_utils.loaders import load_dataset
from vis_utils.plot import plot_scatter
from vis_utils.utils import load_dict, save_dict
from vis_utils.tsne_wrapper import TSNEwrapper
from openTSNE.affinity import Affinities

import umap
from ripser import Rips
from ripser.ripser import get_greedy_perm

from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib import collections  as mc
from matplotlib.colors import Normalize
import matplotlib.cm

from openTSNE.nearest_neighbors import PrecomputedNeighbors
from openTSNE.affinity import PerplexityBasedNN

from vis_utils.utils import kNN_graph, kNN_dists
from scipy.spatial.distance import pdist, squareform
import glasbey
import scipy.sparse
import networkx as nx

from sklearn.decomposition import PCA

from persim import plot_diagrams
from utils.pd_utils import get_life_times
import pandas as pd

from vis_utils.rnaseqTools import geneSelection, sparseload
import pickle
import torch
import urllib.request
import requests
 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUDA_VISIBLE_DEVICES=1


<h4 style="background-color: yellow; padding: 5px;">

to rename the cell culsters of tasic to a yao's way, so that the passing cells of detected features(aka features) are comparable. If you can find the "tasic_to_yao.pkl" file somewhere, no need to run this notebook.

</h4>

In [ ]:
#to download the file....
root_path = get_path("data")
data_dir = os.path.join(root_path, "yao")
file_name = os.path.join(data_dir, "CTX_Hip_anno_SSv4.csv")
url = "https://data.nemoarchive.org/biccn/grant/u19_zeng/zeng/transcriptome/scell/SSv4/mouse/processed/YaoHippo2020/CTX_Hip_anno_SSv4.csv.tar"

with open(file_name, "wb") as f:
    r = requests.get(url)
    f.write(r.content)

In [ ]:
raw_zeng = pd.read_csv(file_name)

/tmp/ipykernel_31959/743725617.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_zeng = pd.read_csv(file_name)


In [ ]:
len(raw_zeng)

73347

In [ ]:
tasic18 = raw_zeng["tasic18_cluster_label"]
yao_new = raw_zeng["cluster_label"]

counts = pd.crosstab(yao_new, tasic18)

In [ ]:
mapped_yao = []
for tasic_label in counts.keys():
    yao_position = counts[tasic_label].argmax()
    mapped_yao.append(counts.index[yao_position])

In [ ]:
tasic_to_yao = dict(zip(list(counts.keys()), mapped_yao))
save_dict(tasic_to_yao, os.path.join(data_dir, "tasic_to_yao.pkl"))